# mt_config — Shared Configuration

Single source of truth for all experiment parameters. Loaded once by `00_main`.

| Setting | Value |
|---|---|
| Catalog | `dev_forge_default.mt_davide` |
| Model | `databricks-meta-llama-3-1-8b-instruct` |
| Tables | 10 `mt_safe_*` pseudonymized analytical tables |

## Architectures
| Architecture | RAG | Multi-Agent | Semantic Delivery |
|---|---|---|---|
| `SAS` | ✗ | ✗ | Direct (full semantic in context) |
| `SAS_RAG` | ✓ | ✗ | Retrieved (top-k chunks) |
| `MAS_RAG` | ✓ | ✓ (static) | Retrieved |
| `DYNAMIC_FILTERED_MAS_RAG` | ✓ | ✓ (adaptive) | Retrieved + filtered |

In [0]:
# =============================================================================
# CONTEXT MODE — single unified context across all architectures
# All architectures query the same mt_safe_* tables and the same semantic
# knowledge source. The delivery mechanism (direct vs retrieved) differs.
# =============================================================================

CONTEXT_MODE = "UNIFIED"

# =============================================================================
# ARCHITECTURE REGISTRY
# Four architectures compared under identical experimental conditions.
# =============================================================================

ARCHITECTURES = {
    "SAS": {
        "use_rag":            False,
        "use_multi_agent":    False,
        "use_reflection":     False,
        "semantic_delivery":  "direct",    # full semantic layer injected into prompt
        "notebook":           "01_Single_Agent_System_SAS",
        "description":        "Single agent. Full semantic context provided directly. No retrieval.",
    },
    "SAS_RAG": {
        "use_rag":            True,
        "use_multi_agent":    False,
        "use_reflection":     False,
        "semantic_delivery":  "retrieved", # top-k semantic chunks retrieved per question
        "notebook":           "02_RAG_Single_Agent",
        "description":        "Single agent. Semantic context retrieved via vector search (top-k=5).",
    },
    "MAS_RAG": {
        "use_rag":            True,
        "use_multi_agent":    True,
        "use_reflection":     False,
        "semantic_delivery":  "retrieved",
        "notebook":           "03_MAS_RAG",
        "description":        "Multi-agent (Planner/Retriever → SQL Analyst → Answer Generator).",
    },
    "DYNAMIC_FILTERED_MAS_RAG": {
        "use_rag":            True,
        "use_multi_agent":    True,
        "use_reflection":     False,
        "semantic_delivery":  "retrieved",
        "notebook":           "04_Dynamic_Filtered_MAS_RAG",
        "description":        "Dynamic filtered MAS: adaptive graph selection, evidence filtering, structured comms.",
    },
}

# Canonical architecture list (no legacy aliases)
CANONICAL_ARCHITECTURES = ["SAS", "SAS_RAG", "MAS_RAG", "DYNAMIC_FILTERED_MAS_RAG"]

# =============================================================================
# SINGLE AGENT PROFILE — Enterprise Data Analyst
# Used for SAS and SAS_RAG. Do NOT vary the profile between runs.
# =============================================================================

ANALYST_PROFILE = (
    "You are an Enterprise Data Analyst responsible for answering structured business questions "
    "using the available data, schema and business knowledge. Analyse the question, identify the "
    "required tables and business rules, create a concise execution plan, generate valid SQL, "
    "execute or validate the query, interpret the returned evidence and provide a concise final "
    "answer. Do not invent values, joins, filters or business definitions. If the available "
    "evidence is insufficient, state the limitation explicitly."
)

# =============================================================================
# EXPERIMENT CONTROL
# =============================================================================

N_REPETITIONS = 1    # runs per question × architecture combination
DEBUG         = False  # True = verbose technical output; False = compact readable output



In [0]:
# =============================================================================
# COMMON SETTINGS — identical across all experiment modes for fair comparison
# =============================================================================
import os as _os
import warnings
warnings.filterwarnings("ignore", message=".*conversion of DecimalType.*")

# =============================================================================
# ENVIRONMENT DETECTION — auto-detect live vs offline mode
# Live mode:    Running in the original Databricks workspace with full access
#               to the experiment catalog, model endpoints, and Delta tables.
# Offline mode: Running elsewhere (another workspace, GitHub import, local).
#               The experiment cannot run; saved results are loaded instead.
# =============================================================================
MT_LIVE_MODE = False
try:
    if 'DATABRICKS_RUNTIME_VERSION' in _os.environ:
        spark.sql(
            "SELECT 1 FROM dev_forge_default.mt_davide.experiment_snapshot_8b_full LIMIT 1"
        ).collect()
        MT_LIVE_MODE = True
except Exception:
    pass  # Databricks env but no access to experiment infrastructure

MT_CATALOG  = "dev_forge_default"
MT_SCHEMA   = "mt_davide"

# Base model — MUST be the same in every mode
MT_MODEL_ENDPOINT = "databricks-meta-llama-3-1-8b-instruct"

# Semantic metadata source
SEMANTIC_CATALOG = "dev_forge_default"
SEMANTIC_SCHEMA  = "relias_care_metadata"

# Logging target — all architectures write here
RESULTS_TABLE    = f"{MT_CATALOG}.{MT_SCHEMA}.experiment_results"
RESULTS_TABLE_V2 = f"{MT_CATALOG}.{MT_SCHEMA}.experiment_results_v2"

# LLM call parameters — same across all modes
LLM_PARAMS = {
    "plan":      {"temperature": 0.3, "max_tokens": 400},
    "sql":       {"temperature": 0.1, "max_tokens": 500},
    "interpret": {"temperature": 0.5, "max_tokens": 600},
    "retrieve":  {"temperature": 0.3, "max_tokens": 300},
    "review":    {"temperature": 0.2, "max_tokens": 400},
}

# Retrieval parameters (RAG and MAS modes)
RETRIEVAL_PARAMS = {
    "embedding_model":    "databricks-gte-large-en",
    "top_k":              5,
    "similarity_threshold": 0.3,
}

# =============================================================================
# REPRODUCIBILITY SEED
# Set to an integer (e.g. 42) for reproducible LLM outputs across runs.
# Set to None for non-deterministic outputs (default during development).
# Toggle: change this value and re-run mt_config.
# =============================================================================
EXPERIMENT_SEED = 42   # Fixed seed for reproducibility

def _seed_kwargs() -> dict:
    """Return {'seed': N} when EXPERIMENT_SEED is set, else empty dict."""
    return {"seed": EXPERIMENT_SEED} if EXPERIMENT_SEED is not None else {}

# =============================================================================
# SHARED LLM CLIENT — 429-resilient, used by all architectures
# =============================================================================
if MT_LIVE_MODE:
    from openai import OpenAI as _OpenAI
    _workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
    _api_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    LLM_CLIENT = _OpenAI(
        api_key=_api_token,
        base_url=f"https://{_workspace_url}/serving-endpoints",
        max_retries=5,
        timeout=120.0,
    )
else:
    LLM_CLIENT = None  # Not available in offline mode

# SQL safety rules — enforced in every mode
SQL_SAFETY_RULES = [
    "Use SELECT only. No INSERT, UPDATE, DELETE, DROP, or CREATE.",
    f"Only query approved mt_safe_* tables in {MT_CATALOG}.{MT_SCHEMA}.",
    f"Always use fully qualified table names: {MT_CATALOG}.{MT_SCHEMA}.mt_safe_<name>.",
    "Use LIMIT for row-level outputs (max 100 rows unless aggregating).",
    "Do NOT query original mt_* tables (without 'safe' prefix).",
    "Do NOT query Bronze, Silver, or Gold source schemas directly.",
    "Do NOT attempt to reverse or expose pseudonymization mappings.",
    "String comparisons are CASE-SENSITIVE. Use exact enum values from context.",
]

In [0]:
# =============================================================================
# APPROVED MT TABLES — Pseudonymized mt_safe_* tables
# These are the ONLY tables any agent is allowed to query.
# =============================================================================

# --- Agent-queryable tables (all modes) ---
APPROVED_SAFE_TABLES = [
    f"{MT_CATALOG}.{MT_SCHEMA}.mt_safe_user",
    f"{MT_CATALOG}.{MT_SCHEMA}.mt_safe_company",
    f"{MT_CATALOG}.{MT_SCHEMA}.mt_safe_assignment_course_progress",
    f"{MT_CATALOG}.{MT_SCHEMA}.mt_safe_course",
    f"{MT_CATALOG}.{MT_SCHEMA}.mt_safe_company_contracts",
    f"{MT_CATALOG}.{MT_SCHEMA}.mt_safe_company_type",
    f"{MT_CATALOG}.{MT_SCHEMA}.mt_safe_certificate",
    f"{MT_CATALOG}.{MT_SCHEMA}.mt_safe_exam_result",
    f"{MT_CATALOG}.{MT_SCHEMA}.mt_safe_assignment",
    f"{MT_CATALOG}.{MT_SCHEMA}.mt_safe_opportunity",
]

# --- Technical metadata table (RAW + SEMANTIC modes) ---
TECHNICAL_METADATA_TABLES = [
    f"{MT_CATALOG}.{MT_SCHEMA}.mt_information_schema",
]

# --- Semantic layer tables (SEMANTIC modes only) ---
SEMANTIC_METADATA_TABLES = [
    f"{MT_CATALOG}.{MT_SCHEMA}.mt_semantic_table_metadata",
    f"{MT_CATALOG}.{MT_SCHEMA}.mt_semantic_column_metadata",
    f"{MT_CATALOG}.{MT_SCHEMA}.mt_semantic_metric_definitions",
    f"{MT_CATALOG}.{MT_SCHEMA}.mt_semantic_join_rules",
    f"{MT_CATALOG}.{MT_SCHEMA}.mt_semantic_business_rules",
]

# --- Composed table sets per mode type ---
RAW_MODE_TABLES = APPROVED_SAFE_TABLES + TECHNICAL_METADATA_TABLES
SEMANTIC_MODE_TABLES = APPROVED_SAFE_TABLES + TECHNICAL_METADATA_TABLES + SEMANTIC_METADATA_TABLES

# --- Join relationships (pseudonymization groups) ---
MT_JOIN_RULES = [
    {"group": "user",       "columns": ["mt_safe_user.user_id", "mt_safe_user.u_id", "mt_safe_user.exam_userId", "mt_safe_assignment_course_progress.user_id", "mt_safe_certificate.userId", "mt_safe_exam_result.exam_userId"]},
    {"group": "company",    "columns": ["mt_safe_assignment.company_ID1", "mt_safe_company.company_ID", "mt_safe_company_contracts.companyId"]},
    {"group": "course",     "columns": ["mt_safe_assignment_course_progress.course_id", "mt_safe_certificate.courseId", "mt_safe_course.course_id1", "mt_safe_user.course_id", "mt_safe_user.courseId", "mt_safe_user.exam_courseId"]},
    {"group": "assignment", "columns": ["mt_safe_assignment.assignment_ID", "mt_safe_assignment_course_progress.assignment_id1", "mt_safe_certificate.assignmentId", "mt_safe_exam_result.exam_assignmentId", "mt_safe_user.assignment_id1", "mt_safe_user.ass_id", "mt_safe_user.exam_assignmentId"]},
    {"group": "certificate", "columns": ["mt_safe_certificate.id", "mt_safe_user.id"]},
    {"group": "exam",       "columns": ["mt_safe_exam_result.examId", "mt_safe_user.examId"]},
    {"group": "company_type", "columns": ["mt_safe_company.subTypeId", "mt_safe_company_type.type_id"]},
    {"group": "contract",   "columns": ["mt_safe_company_contracts.id"]},
    {"group": "opportunity", "columns": ["mt_safe_opportunity.opportunity_id"]},
    {"group": "account",    "columns": ["mt_safe_opportunity.account_id"]},
]

In [0]:
# =============================================================================
# CLAIM-LEVEL EVALUATION PARAMS (RAGChecker-inspired)
# Fixed evaluator configuration — same models/prompts for every architecture.
# =============================================================================

CLAIM_EVALUATION_PARAMS = {
    "extractor_model": MT_MODEL_ENDPOINT,  # fixed evaluator endpoint
    "checker_model": MT_MODEL_ENDPOINT,    # fixed evaluator endpoint
    "temperature": 0.0,
    "top_p": 1.0,
    "max_tokens": 1000,
    "max_retries": 2,
    "prompt_version": "claim_eval_v1",
    # --- Tolerance thresholds (relaxed for practical evaluation) ---
    # Rationale: agents compute metrics via different SQL approaches (e.g.,
    # rounding at different stages, different denominators). Overly tight
    # tolerances penalise correct-in-spirit answers.
    "default_relative_tolerance": 0.05,       # 5% relative (was 1%)
    "default_absolute_tolerance": 1.0,         # absolute ±1 (was 1e-6)
    "percentage_point_tolerance": 2.0,          # ±2 pp (was 0.5 pp)
    "count_relative_tolerance": 0.02,           # ±2% for counts (was exact)
}

# Entity/metric aliases for normalization
CLAIM_ALIASES = {
    "entities": {
        "company_id2": ["company_id", "company id", "companyid"],
        "user_id": ["userid", "user id"],
    },
    "metrics": {
        "certificate count": ["certificates", "total certificates", "cert count"],
        "distinct users": ["unique users", "distinct user count"],
        "on-time completion rate": ["on-time rate", "compliance rate"],
    },
    "units": {
        "%": ["percent", "percentage", "pct"],
        "eur": ["euro", "euros", "€"],
    },
}

In [0]:
# =============================================================================
# CONTEXT TABLE SETS — What each architecture sees
# In UNIFIED mode, all architectures get the same queryable + metadata tables.
# The delivery mechanism (direct vs. retrieved) differs by architecture.
# =============================================================================

def get_tables_for_architecture(arch: str) -> dict:
    """
    Return the table sets available to a given architecture.
    
    Returns dict with:
      - queryable_tables: tables the agent can SELECT from
      - metadata_tables:  tables used to build the agent's context
      - semantic_delivery: how the semantic layer is provided
    """
    if arch not in ARCHITECTURES:
        raise ValueError(f"Unknown architecture: {arch}. Valid: {list(ARCHITECTURES.keys())}")
    
    config = ARCHITECTURES[arch]
    
    return {
        "queryable_tables":   APPROVED_SAFE_TABLES,
        "metadata_tables":    TECHNICAL_METADATA_TABLES + SEMANTIC_METADATA_TABLES,
        "semantic_delivery":  config["semantic_delivery"],
    }


# =============================================================================
# PRINT CONFIG SUMMARY
# =============================================================================

# Only print banner on first load (suppress when sub-notebooks re-run mt_config)
if '_MT_CONFIG_LOADED' not in dir():
    _MT_CONFIG_LOADED = True
    _mode_label = "🟢 LIVE" if MT_LIVE_MODE else "🟡 OFFLINE (saved results only)"
    print(f"✓ mt_config | {MT_CATALOG}.{MT_SCHEMA} | {MT_MODEL_ENDPOINT}")
    print(f"  Mode: {_mode_label}")

✓ mt_config loaded
  Experiment modes:    ['SAS_SQL_RAW', 'SAS_SQL_SEMANTIC', 'SAS_RAG_SEMANTIC', 'MAS_RAG_SEMANTIC', 'FILTERED_MAS_RAG_SEMANTIC']
  MT catalog:          dev_forge_default.mt_davide
  Model endpoint:      databricks-meta-llama-3-1-8b-instruct
  Results table:       dev_forge_default.mt_davide.experiment_results
  Safe tables:         10 (mt_safe_*)
  Technical metadata:  1 (mt_information_schema)
  Semantic metadata:   5 (mt_semantic_*)
  Join groups:         10

  RAW mode tables:      11 = 10 safe + 1 metadata
  SEMANTIC mode tables: 16 = 10 safe + 1 metadata + 5 semantic


In [0]:
# =============================================================================
# MAS AGENT PROFILES — 5 Expertise-Based Professionals
# =============================================================================
# Each profile represents a genuine professional with unique domain knowledge.
# Every agent CAN solve the problem end-to-end, but contributes their specific
# expertise to the collaborative pipeline. Used by both MAS_RAG (static, all 5)
# and DYNAMIC_FILTERED_MAS_RAG (FFN-gated subset).
# =============================================================================

AGENT_PROFILES = {
    "DOMAIN_EXPERT": {
        "role": "business_domain",
        "description": "Healthcare compliance specialist. Interprets business questions using Relias domain knowledge.",
        "llm_params": LLM_PARAMS["plan"],
        "system_prompt": (
            "You are a Healthcare Compliance Specialist with deep knowledge of the Relias "
            "e-learning certification platform. You understand:\n"
            "- Pflichtzertifikat (mandatory certificate) vs Wahlzertifikat (elective certificate)\n"
            "- Pflichtkurs (mandatory course) vs Wahlkurs (elective course)\n"
            "- On-time compliance: a certificate is on-time when completedOn_ts <= deadline_assignment\n"
            "- Company types: MVZ, Care, Baercare (linked via subTypeId → type_id)\n"
            "- Company statuses: ACTIVE, INACTIVE, TRIAL\n"
            "- The assignment-based hierarchy: users belong to assignments, assignments belong to companies\n"
            "- A certificate represents a completed course; certificate_type indicates mandatory/elective\n\n"
            "Given a business question and data context, you:\n"
            "1. Interpret what the question truly asks in business terms\n"
            "2. Identify which business rules, filters, and definitions apply\n"
            "3. Flag any ambiguities or domain-specific considerations\n"
            "4. Write SQL if needed, produce a complete answer\n\n"
            "If the question CANNOT be answered from the available data, respond with CANNOT_ANSWER.\n"
            "Do not invent business rules or definitions not present in your knowledge."
        ),
    },
    "DATA_ENGINEER": {
        "role": "schema_navigation",
        "description": "Data model expert. Knows table schemas, join paths, alias columns, and data types.",
        "llm_params": LLM_PARAMS["plan"],
        "system_prompt": (
            "You are a Data Engineer who built and maintains the mt_safe_* data model. "
            "You have expert knowledge of:\n"
            "- mt_safe_user has 32 columns with DUPLICATE ALIASES:\n"
            "  • user_id is the PRIMARY key (u_id and exam_userId are aliases of user_id)\n"
            "  • course_id is PRIMARY (courseId and exam_courseId are aliases)\n"
            "  • assignment_id1 is PRIMARY (ass_id and exam_assignmentId are aliases)\n"
            "  • 'id' column is a Certificate ID (CERT_xxx), NOT a user ID\n"
            "  • completedOn_ts is the canonical timestamp (completedOn is alias)\n"
            "- mt_safe_user has NO company_ID column — join through mt_safe_assignment\n"
            "- Canonical join path for company:\n"
            "  mt_safe_user.assignment_id1 → mt_safe_assignment.assignment_ID →\n"
            "  mt_safe_assignment.company_ID1 → mt_safe_company.company_ID\n"
            "- Company type join: mt_safe_company.subTypeId → mt_safe_company_type.type_id\n"
            "- course_type column exists in mt_safe_assignment_course_progress, NOT mt_safe_course\n"
            "- mt_safe_course uses course_id1 (not course_id)\n\n"
            "Given a question and previous agents' analysis, you:\n"
            "1. Identify the exact tables needed\n"
            "2. Specify the correct join paths with column names\n"
            "3. Flag alias columns to avoid (always prefer PRIMARY columns)\n"
            "4. Write SQL if needed, produce a complete answer\n\n"
            "Always use fully qualified names: dev_forge_default.mt_davide.mt_safe_<name>\n"
            "Prevent wrong joins by being explicit about which column from which table."
        ),
    },
    "SQL_DEVELOPER": {
        "role": "sql_generation",
        "description": "Senior SQL developer. Writes precise, correct Databricks SQL queries.",
        "llm_params": LLM_PARAMS["sql"],
        "system_prompt": (
            "You are a Senior SQL Developer specialized in Databricks SQL. You write "
            "precise, optimized queries following strict rules:\n"
            "- SELECT only. No INSERT, UPDATE, DELETE, DROP, CREATE\n"
            "- Fully qualified names: dev_forge_default.mt_davide.mt_safe_<name>\n"
            "- CASE-SENSITIVE string comparisons (use exact enum values)\n"
            "- Correct window functions: ROW_NUMBER() OVER, LAG/LEAD, SUM OVER\n"
            "- NEVER nest aggregate functions (use CTEs or subqueries instead)\n"
            "- Use CAST(CASE WHEN condition THEN 1 ELSE 0 END AS DOUBLE) for rates, not AVG(boolean)\n"
            "- All non-aggregated columns MUST appear in GROUP BY\n"
            "- Date operations: YEAR(), MONTH(), proper date type comparisons\n"
            "- LIMIT for row-level outputs (max 100), no LIMIT for aggregations\n"
            "- Prefer CTEs (WITH clause) for complex multi-step queries\n\n"
            "Given a question and previous agents' analysis (business rules, join paths), you:\n"
            "1. Produce a single correct SQL query incorporating their guidance\n"
            "2. Return ONLY the raw SQL (no markdown fencing, no explanations before/after)\n"
            "3. If the question cannot be answered from the available data, respond with CANNOT_ANSWER.\n\n"
            "Use the schema context and join information from previous agents to write correct SQL."
        ),
    },
    "QUANTITATIVE_ANALYST": {
        "role": "numeric_interpretation",
        "description": "Metrics specialist. Interprets SQL results, computes rates, validates numbers.",
        "llm_params": LLM_PARAMS["interpret"],
        "system_prompt": (
            "You are a Quantitative Analyst specialized in business metrics computation. "
            "You excel at:\n"
            "- Rate calculations: on_time_count / total_count (not AVG of boolean)\n"
            "- Percentage formatting: present as XX.X% with one decimal\n"
            "- Month-over-month growth: (current - previous) / previous × 100\n"
            "- Cohort analysis: grouping by creation period, tracking behavior\n"
            "- Small sample awareness: flag when N < 30 affects reliability\n"
            "- Drop detection: identify consecutive period declines\n"
            "- Ranking: distinguish between DENSE_RANK, ROW_NUMBER, TOP-N\n\n"
            "Given a question, SQL results, and previous agents' context, you:\n"
            "1. Interpret the SQL output into meaningful business numbers\n"
            "2. Compute any derived metrics (rates, growth, comparisons)\n"
            "3. Flag statistical caveats (small samples, outliers, bias)\n"
            "4. Produce a precise numerical answer with proper formatting\n\n"
            "CRITICAL: Every number you state MUST come from the SQL results provided.\n"
            "If the question cannot be answered from the available data, respond with CANNOT_ANSWER.\n"
            "Do NOT invent plausible-sounding numbers."
        ),
    },
    "QUALITY_AUDITOR": {
        "role": "validation",
        "description": "Data quality auditor. Validates grounding, catches hallucinations, produces final answer.",
        "llm_params": LLM_PARAMS["interpret"],
        "system_prompt": (
            "You are a Data Quality Auditor. Your job is to validate whether an answer "
            "is grounded in actual evidence and produce the final verified response.\n\n"
            "You check:\n"
            "- Every number in the answer MUST trace back to SQL output or retrieved evidence\n"
            "- Plausibility: rates must be 0-100%, counts must be non-negative\n"
            "- Completeness: does the answer address ALL parts of the question?\n\n"
            "CRITICAL ABSTENTION RULES — respond with CANNOT_ANSWER ONLY when:\n"
            "  a) The question requires data that DOES NOT EXIST in the mt_safe_* schema\n"
            "     (e.g. revenue, login timestamps, satisfaction/NPS scores, geographic location,\n"
            "      instructor data, training budgets — these columns/tables do not exist)\n"
            "  b) SQL execution FAILED on all attempts AND no alternative query is possible\n"
            "In ALL other cases, you MUST produce a final answer:\n"
            "- If SQL succeeded and returned rows: answer using those numbers\n"
            "- If SQL returned 0 rows: report that the query returned no matching data\n"
            "- If some numbers look unexpected: present them with appropriate caveats\n"
            "- If you detect hallucinated numbers (not from SQL output): remove them and\n"
            "  answer using ONLY the verified numbers from SQL results\n\n"
            "Your output IS the final answer. Be concise, cite specific numbers from evidence, "
            "and never add information not present in the SQL results or retrieved context."
        ),
    },
}

# Agent execution order for static MAS (all 5 always participate)
MAS_AGENT_ORDER = ["DOMAIN_EXPERT", "DATA_ENGINEER", "SQL_DEVELOPER", "QUANTITATIVE_ANALYST", "QUALITY_AUDITOR"]

# V3 logging table
V3_RUNS_TABLE = f"{MT_CATALOG}.{MT_SCHEMA}.experiment_runs_v3"

print(f"✓ AGENT_PROFILES defined: {list(AGENT_PROFILES.keys())}")
print(f"  MAS static order: {' → '.join(MAS_AGENT_ORDER)}")